In [25]:
import yfinance as yf
import pandas as pd
import requests
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
data = "ZL=F,ZS=F"

#### Download data from Yahoo Finance

In [46]:
data_recent = yf.download(data, start="2025-09-01", end="2026-01-04", interval="1d")

[*********************100%***********************]  2 of 2 completed


In [47]:
data_recent.dropna(inplace = True)

#### Calculate ATR for Position Sizing

In [48]:
#Reorentate the stacked headers into columns
data_recent_atr = data_recent.stack(level=1).reset_index(level=0).rename({'level_1': 'Ticker'}, axis=1)
data_recent_atr.reset_index(inplace=True)
data_recent_atr.rename(columns = {'index':'Ticker'}, inplace=True)
data_recent_atr.sort_values(['Ticker', 'Date'], inplace = True)

/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_95883/979187486.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data_recent_atr = data_recent.stack(level=1).reset_index(level=0).rename({'level_1': 'Ticker'}, axis=1)


#### Calculate Returns

In [49]:
data_recent = data_recent["Close"]


In [50]:

data_recent.reset_index(inplace = True)


In [51]:
data_recent.head()

Ticker,Date,ZL=F,ZS=F
0,2025-09-02,51.970001,1025.75
1,2025-09-03,51.160000,1016.00
2,2025-09-04,51.230000,1012.00
3,2025-09-05,50.560001,1006.50
4,2025-09-08,50.730000,1013.50


In [52]:
cols = ["ZL=F", "ZS=F"]

for colname in cols:
    
    data_recent[colname+ "_return"] = (data_recent[colname] - data_recent[colname].shift(1)) / data_recent[colname].shift(1)

In [53]:
data_recent.fillna(0, inplace = True)

In [54]:
stock1 = data_recent.columns[1]
stock2 = data_recent.columns[2]
market_name = data_recent.columns[3]

In [55]:
data_recent.head()

Ticker,Date,ZL=F,ZS=F,ZL=F_return,ZS=F_return
0,2025-09-02,51.970001,1025.75,0.000000,0.000000
1,2025-09-03,51.160000,1016.00,-0.015586,-0.009505
2,2025-09-04,51.230000,1012.00,0.001368,-0.003937
3,2025-09-05,50.560001,1006.50,-0.013078,-0.005435
4,2025-09-08,50.730000,1013.50,0.003362,0.006955


In [56]:
data_recent[["ZL=F_return", "ZS=F_return"]]

Ticker,ZL=F_return,ZS=F_return
0,0.000000,0.000000
1,-0.015586,-0.009505
2,0.001368,-0.003937
3,-0.013078,-0.005435
4,0.003362,0.006955
...,...,...
81,-0.006323,-0.004232
82,0.001231,-0.008737
83,0.003075,-0.003097
84,-0.017576,-0.015054


#### Variance and Covariance of Returns

In [57]:
covariance = data_recent[["ZL=F_return", "ZS=F_return"]].cov()

var_zs = data_recent["ZS=F_return"].var()

market_var = covariance / var_zs

### Beta Weighting

Beta weighting is the stock's covariance with the overall market per unit of market risk

Beta = COV (stock) /Var (market)

Beta is a measure of sytematic risk.  If Beta is 1.10 then the stock has 10% more risk than the market.

Use the S&P 500 to measure the overall market

If market is expected to be up/down 20% over a period then the stock is expected to be up/down 22%


#### CAPM: Capital Asset Pricing Model

This model asserts that a stock's return is based on its Beta

<img src = "CAPM.png" alt="CAPM Model" width=300 height =300 >


#### Beta Nutrality

Beta stock1 X Price stock 1 X nShares stock 1 == Beta stock2 X Price stock 2 X nShares stock 2

i.e the beta of apple X the market value of apple should equal the beta of msft X the market value os msft

#### Calculating Beta

In [58]:
# Step 1: Covariance between the two returns
cov_zl_zs = data_recent["ZL=F_return"].cov(data_recent["ZS=F_return"])

# Step 2: Variance of your anchor leg (ZL, since you're going long ZL)
var_zl = data_recent["ZL=F_return"].var()

# Step 3: Hedge ratio (this IS the "beta" for pairs trading)
hedge_ratio = cov_zl_zs / var_zl

# Contract specifications
zl_contract_size = 60000  # pounds
zs_contract_size = 5000   # bushels

# Current prices (Yahoo returns CENTS, convert to dollars)
zl_price = data_recent["ZL=F"].iloc[-1] / 100  # cents/lb → $/lb
zs_price = data_recent["ZS=F"].iloc[-1] / 100  # cents/bu → $/bu

# Contract dollar values
zl_contract_value = zl_price * zl_contract_size
zs_contract_value = zs_price * zs_contract_size

print(f"\nPrices (converted to $):")
print(f"  ZL: ${zl_price:.4f}/lb")
print(f"  ZS: ${zs_price:.4f}/bushel")

print(f"\nContract Values:")
print(f"  ZL contract: ${zl_contract_value:,.0f}")
print(f"  ZS contract: ${zs_contract_value:,.0f}")

# Dollar-adjusted hedge ratio
dollar_adjusted_ratio = hedge_ratio * (zl_contract_value / zs_contract_value)


print(f"\nReturn-based hedge ratio: {hedge_ratio:.4f}")
print(f"Dollar-adjusted hedge ratio: {dollar_adjusted_ratio:.4f}")
print(f"\nFor 1 ZL contract LONG (${zl_contract_value:,.0f}):")
print(f"  SHORT {dollar_adjusted_ratio:.2f} ZS contracts (${dollar_adjusted_ratio * zs_contract_value:,.0f})")


Prices (converted to $):
  ZL: $0.4930/lb
  ZS: $10.4575/bushel

Contract Values:
  ZL contract: $29,580
  ZS contract: $52,288

Return-based hedge ratio: 0.4970
Dollar-adjusted hedge ratio: 0.2812

For 1 ZL contract LONG ($29,580):
  SHORT 0.28 ZS contracts ($14,701)


In [59]:
# Your risk tolerance
risk_amount = 150  # dollars you're willing to lose

# Contract multipliers (to convert price moves to $)
zl_multiplier = 600    # $600 per 1 cent move (60,000 lbs × $0.01)
zs_multiplier = 50     # $50 per 1 cent move (5,000 bu × $0.01)

# Calculate ATR for each leg (14-period)
zl_atr = data_recent_atr[data_recent_atr['Ticker'] == 'ZL=F']['Close'].pct_change().abs().rolling(14).mean().iloc[-1]
zs_atr = data_recent_atr[data_recent_atr['Ticker'] == 'ZS=F']['Close'].pct_change().abs().rolling(14).mean().iloc[-1]

# Or use simple price ATR (more intuitive):
zl_prices = data_recent["ZL=F"]
zs_prices = data_recent["ZS=F"]

zl_atr_price = (zl_prices.diff().abs()).rolling(14).mean().iloc[-1]  # in cents
zs_atr_price = (zs_prices.diff().abs()).rolling(14).mean().iloc[-1]  # in cents

# Dollar ATR per contract
zl_dollar_atr = zl_atr_price * zl_multiplier
zs_dollar_atr = zs_atr_price * zs_multiplier

print(f"ZL ATR: {zl_atr_price:.2f} cents = ${zl_dollar_atr:.0f}/contract")
print(f"ZS ATR: {zs_atr_price:.2f} cents = ${zs_dollar_atr:.0f}/contract")

# Combined spread risk (conservative: assume both legs move against you, reduced by correlation)
correlation = data_recent["ZL=F_return"].corr(data_recent["ZS=F_return"])
spread_dollar_atr = np.sqrt(zl_dollar_atr**2 + (dollar_adjusted_ratio * zs_dollar_atr)**2 
                            - 2 * correlation * zl_dollar_atr * dollar_adjusted_ratio * zs_dollar_atr)

print(f"\nCorrelation: {correlation:.3f}")
print(f"Spread ATR (1 unit): ${spread_dollar_atr:.0f}")

# Position sizing: risk 2x ATR as stop
stop_multiple = 2  # how many ATRs for your stop
stop_distance = spread_dollar_atr * stop_multiple

# Number of spread units you can trade
spread_units = risk_amount / stop_distance

print(f"\n=== POSITION SIZE (risking ${risk_amount}) ===")
print(f"Stop: {stop_multiple}x ATR = ${stop_distance:.0f}")
print(f"Spread units: {spread_units:.2f}")
print(f"\nTrade:")
print(f"  LONG  {spread_units:.2f} ZL contracts")
print(f"  SHORT {spread_units * dollar_adjusted_ratio:.2f} ZS contracts")

ZL ATR: 0.52 cents = $309/contract
ZS ATR: 7.84 cents = $392/contract

Correlation: 0.595
Spread ATR (1 unit): $259

=== POSITION SIZE (risking $150) ===
Stop: 2x ATR = $519
Spread units: 0.29

Trade:
  LONG  0.29 ZL contracts
  SHORT 0.08 ZS contracts


In [60]:
# =============================================================================
# POSITION SIZING WITH DOLLAR VALUES
# =============================================================================

# Your risk tolerance
risk_amount = 150  # dollars you're willing to lose

# Contract multipliers (to convert price moves to $)
zl_multiplier = 600    # $600 per 1 cent move (60,000 lbs × $0.01)
zs_multiplier = 50     # $50 per 1 cent move (5,000 bu × $0.01)

# Current prices and contract values
zl_price = data_recent["ZL=F"].iloc[-1]  # in cents
zs_price = data_recent["ZS=F"].iloc[-1]  # in cents
zl_contract_value = (zl_price / 100) * 60000  # convert to $ 
zs_contract_value = (zs_price / 100) * 5000   # convert to $

# ATR calculation (price-based, in cents)
zl_atr_price = (data_recent["ZL=F"].diff().abs()).rolling(14).mean().iloc[-1]
zs_atr_price = (data_recent["ZS=F"].diff().abs()).rolling(14).mean().iloc[-1]

# Dollar ATR per contract
zl_dollar_atr = zl_atr_price * zl_multiplier
zs_dollar_atr = zs_atr_price * zs_multiplier

# Combined spread risk
correlation = data_recent["ZL=F_return"].corr(data_recent["ZS=F_return"])
spread_dollar_atr = np.sqrt(zl_dollar_atr**2 + (dollar_adjusted_ratio * zs_dollar_atr)**2 
                            - 2 * correlation * zl_dollar_atr * dollar_adjusted_ratio * zs_dollar_atr)

# Position sizing
stop_multiple = 2
stop_distance = spread_dollar_atr * stop_multiple
spread_units = risk_amount / stop_distance

# Calculate dollar exposures
zl_contracts = spread_units
zs_contracts = spread_units * dollar_adjusted_ratio
zl_dollar_exposure = zl_contracts * zl_contract_value
zs_dollar_exposure = zs_contracts * zs_contract_value

print("=" * 50)
print(f"POSITION SIZE (risking ${risk_amount})")
print("=" * 50)
print(f"\nContract Values:")
print(f"  1 ZL contract = ${zl_contract_value:,.0f}")
print(f"  1 ZS contract = ${zs_contract_value:,.0f}")

print(f"\nYour Position:")
print(f"  LONG  {zl_contracts:.2f} ZL = ${zl_dollar_exposure:,.0f} exposure")
print(f"  SHORT {zs_contracts:.2f} ZS = ${zs_dollar_exposure:,.0f} exposure")
print(f"  Net exposure: ${abs(zl_dollar_exposure - zs_dollar_exposure):,.0f}")

print(f"\n" + "=" * 50)
print("STOP-LOSS LEVELS")
print("=" * 50)

# Stop loss prices (2 ATR adverse move)
zl_stop = zl_price - (zl_atr_price * stop_multiple)  # long ZL, stop below
zs_stop = zs_price + (zs_atr_price * stop_multiple)  # short ZS, stop above

print(f"\nIndividual leg stops (for reference):")
print(f"  ZL entry: {zl_price:.2f} → Stop: {zl_stop:.2f} ({stop_multiple}x ATR = {zl_atr_price * stop_multiple:.2f} cents)")
print(f"  ZS entry: {zs_price:.2f} → Stop: {zs_stop:.2f} ({stop_multiple}x ATR = {zs_atr_price * stop_multiple:.2f} cents)")

# Spread-based stop
entry_ratio = zl_price / zs_price
spread_atr = np.sqrt((zl_atr_price/zl_price)**2 + (zs_atr_price/zs_price)**2 
                      - 2*correlation*(zl_atr_price/zl_price)*(zs_atr_price/zs_price)) * entry_ratio
stop_ratio = entry_ratio - (spread_atr * stop_multiple)

print(f"\nSpread-based stop (RECOMMENDED):")
print(f"  Entry ratio (ZL/ZS): {entry_ratio:.4f}")
print(f"  Stop ratio:          {stop_ratio:.4f}")
print(f"  Exit if ZL/ZS falls below {stop_ratio:.4f}")

POSITION SIZE (risking $150)

Contract Values:
  1 ZL contract = $29,580
  1 ZS contract = $52,288

Your Position:
  LONG  0.29 ZL = $8,552 exposure
  SHORT 0.08 ZS = $4,250 exposure
  Net exposure: $4,302

STOP-LOSS LEVELS

Individual leg stops (for reference):
  ZL entry: 49.30 → Stop: 48.27 (2x ATR = 1.03 cents)
  ZS entry: 1045.75 → Stop: 1061.43 (2x ATR = 15.68 cents)

Spread-based stop (RECOMMENDED):
  Entry ratio (ZL/ZS): 0.0471
  Stop ratio:          0.0463
  Exit if ZL/ZS falls below 0.0463


### Kalman Filtering

Kalman filtering is a dynamic way of weighting the pairs trade, it requires the trader to update it and adjust positions as conditions change. It has covariance baked in and is weighted forward. It doesnt require rolling windows.
Beta weighting is more static but takes into account the benchmark index and removes it. Kalman is seen as harder and more complex to implement although seems of equal effort to me.

In [ ]:
himx = data_recent["HIMX"]
apple = data_recent["AAPL"]

In [ ]:
print(himx.isna().sum())

In [ ]:
from pykalman import KalmanFilter

In [ ]:
#taken from coursera
kf = KalmanFilter(transition_matrices = [1],
                  observation_matrices = [1],
                  initial_state_mean = 0, #have to have a start value, this changes later
                  initial_state_covariance = 1,
                  observation_covariance = 1,
                  transition_covariance = 0.01)

In [ ]:
#Use values of price to get rolling meeans nased on the kalman equation which attempts to remove noise from the price data
state_means,_ = kf.filter(himx.values)
state_means = pd.Series(state_means.flatten(), index=himx.index) 

In [ ]:
#compute rolling average
mean50 = himx.rolling(window =50).mean()
mean100 = himx.rolling(window =100).mean()

#plot orignal data and means
plt.plot(state_means)
plt.plot(himx)
plt.plot(mean50)
plt.plot(mean100)
plt.title("Experimenting with Kalman mean to remove price noise")
plt.legend(['Kalman Estimate', "Himx Price", "50 MA", "100 MA"])
plt.xlabel("Day")
plt.ylabel('Price')


In [ ]:
spread = apple - himx

In [ ]:
#Using Kalman as a hedge ratio instead of beta weighting

# Reshape spread.values to a 2D array (n_samples, 1)
spread_reshaped = spread.values.reshape(-1, 1)

# Calculate the hedge ratio using the Kalman filter
kf = KalmanFilter(transition_matrices=[1], observation_matrices=np.eye(1), observation_covariance=1)
state_means, _ = kf.filter(spread_reshaped)
hedge_ratio = state_means.flatten()

# Calculate the number of shares to trade for each stock based on a capital of $10,000
capital_kf = 10000  # USD
aapl_price = apple.iloc[-1]  # Assuming last price for AAPL
himx_price = himx.iloc[-1]  # Assuming last price for HIMX

# Calculate the number of shares for each stock based on the hedge ratio and capital
aapl_shares = (capital_kf / (aapl_price + hedge_ratio[-1] * himx_price))
himx_shares = hedge_ratio[-1] * aapl_shares

print(f"Estimated number of AAPL shares to trade: {aapl_shares:.2f}")
print(f"Estimated number of HIMX shares to trade: {himx_shares:.2f}")


In [ ]:
#take a look at pairs selection from the labe and reprodice the code here, it should be about the beta weighting.
#find negatively correlated stocks
#Find negatively correlated funds.
